# 01 - EDA clínico

Exploración de la ficha clínica: dimensiones, tipos, distribuciones, comparación sano vs. Alzheimer.

**Pregunta que responde este notebook:** ¿qué variables clínicas muestran diferencias entre pacientes
sanos y pacientes con Alzheimer en esta cohorte sintética?

**Salidas principales**
- `results/tables/resumen_numericas_por_grupo.csv` — media, mediana, mín, máx y tamaño de efecto
- `results/tables/resumen_binarias_por_grupo.csv` — prevalencias y diferencia en puntos
- `results/tables/variables_seleccionadas.csv` — las 3–5 variables con mayor diferencia
- `results/tables/clinica_para_integracion.csv` — insumo del notebook 04
- `results/figures/` — figuras para la presentación

> Los datos son **sintéticos** y de uso docente. Las asociaciones observadas no son evidencia clínica.


## 0. Configuración

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

sns.set_theme(style="whitegrid", context="notebook")
PALETA = {"Sano": "#4C72B0", "Alzheimer": "#DD8452"}

RANDOM_STATE = 42

### Rutas

El notebook vive en `notebooks/`, así que los datos y resultados se buscan en el nivel superior.
Si tus archivos están en otra parte, edita `DIR_DATOS`.

In [ ]:
DIR_BASE      = Path("..").resolve()
DIR_DATOS     = DIR_BASE / "data"
DIR_RESULTADOS = DIR_BASE / "results"

DIR_FIGURAS = DIR_RESULTADOS / "figures"
DIR_TABLAS  = DIR_RESULTADOS / "tables"

for sub in ["distribuciones", "categoricas", "correlacion", "proyecciones", "resumen"]:
    (DIR_FIGURAS / sub).mkdir(parents=True, exist_ok=True)
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

ARCHIVO_CLINICO = DIR_DATOS / "synthetic_alzheimer_patients_1000.csv"
ARCHIVO_META    = DIR_DATOS / "metadata.json"

print("Datos:     ", ARCHIVO_CLINICO, "->", ARCHIVO_CLINICO.exists())
print("Metadata:  ", ARCHIVO_META, "->", ARCHIVO_META.exists())
print("Resultados:", DIR_RESULTADOS)

In [ ]:
def guardar_figura(nombre, subcarpeta="resumen"):
    """Guarda la figura activa en results/figures/<subcarpeta>/ a 300 dpi."""
    ruta = DIR_FIGURAS / subcarpeta / f"{nombre}.png"
    plt.savefig(ruta, dpi=300, bbox_inches="tight")
    return ruta


def guardar_tabla(df, nombre, index=False):
    """Guarda una tabla en results/tables/ y devuelve la ruta."""
    ruta = DIR_TABLAS / f"{nombre}.csv"
    df.to_csv(ruta, index=index)
    return ruta

---
## 1. Carga y control de calidad

La guía (sección 4.1) pide revisar dimensiones, tipos de variables, valores faltantes y duplicados
antes de cualquier análisis. Es la parte menos vistosa y la primera que preguntan en la defensa.

In [ ]:
df = pd.read_csv(ARCHIVO_CLINICO)

with open(ARCHIVO_META, "r", encoding="utf-8") as f:
    metadata = json.load(f)

LABEL   = metadata["label"]        # "alzheimer"
COL_ID  = metadata["col_id"]       # "patient_id"
NUMERICAS = metadata["numerical_column"]
BINARIAS  = metadata["categorical_column"]

print(f"Pacientes (filas):  {df.shape[0]}")
print(f"Variables (columnas): {df.shape[1]}")
print(f"Numéricas: {len(NUMERICAS)} | Binarias: {len(BINARIAS)} | Id: {COL_ID} | Etiqueta: {LABEL}")

df.head()

In [ ]:
# Tipos de dato
tipos = (df.dtypes.rename("tipo").to_frame()
         .assign(grupo=lambda t: np.select(
             [t.index == COL_ID, t.index == LABEL,
              t.index.isin(NUMERICAS), t.index.isin(BINARIAS)],
             ["identificador", "diagnostico", "numerica", "binaria"],
             default="sin_clasificar")))

print(tipos["grupo"].value_counts().to_string())
tipos

In [ ]:
# Faltantes y duplicados
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0]

print("Valores faltantes:")
print(faltantes.to_string() if len(faltantes) else "  Ninguno")

print(f"\nFilas completamente duplicadas: {df.duplicated().sum()}")
print(f"{COL_ID} duplicados: {df[COL_ID].duplicated().sum()}")

# Las binarias deberían contener solo 0 y 1
no_binarias = {c: sorted(df[c].dropna().unique())
               for c in BINARIAS if not set(df[c].dropna().unique()) <= {0, 1}}
print("\nColumnas declaradas binarias con valores fuera de {0,1}:",
      no_binarias if no_binarias else "ninguna")

# Rangos plausibles de las numéricas
df[NUMERICAS].describe().T[["min", "max", "mean", "50%"]].round(2)

---
## 2. Balance de la cohorte

Cuántos pacientes hay en cada grupo. Es el número que abre la diapositiva de datos.

In [ ]:
df["diagnostico"] = df[LABEL].map({0: "Sano", 1: "Alzheimer"})

conteo = df["diagnostico"].value_counts()
porcentaje = df["diagnostico"].value_counts(normalize=True).mul(100).round(1)

balance = pd.DataFrame({"n_pacientes": conteo, "porcentaje": porcentaje})
display(balance)

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=balance.index, y=balance["n_pacientes"], palette=PALETA, ax=ax)
for i, v in enumerate(balance["n_pacientes"]):
    ax.text(i, v + 8, f"{v}\n({balance['porcentaje'].iloc[i]}%)", ha="center", fontsize=10)
ax.set(xlabel="", ylabel="Número de pacientes", title="Balance de la cohorte")
ax.set_ylim(0, balance["n_pacientes"].max() * 1.18)
plt.tight_layout()
guardar_figura("balance_cohorte")
plt.show()

**Nota para la presentación:** la cohorte está desbalanceada (~2 controles por cada caso).
Eso no impide comparar, pero sí obliga a usar porcentajes en vez de conteos absolutos al
comparar prevalencias entre grupos.

---
## 3. Tabla resumen de variables numéricas

La guía pide media, mediana, mínimo y máximo por grupo. Agregamos el **tamaño de efecto
(d de Cohen)** para poder ordenar las variables por magnitud de la diferencia, en vez de
elegirlas a ojo.

$$d = \frac{\bar{x}_{Alzheimer} - \bar{x}_{sano}}{s_{combinada}}$$

Referencia: |d| ≈ 0.2 pequeño, 0.5 mediano, 0.8 grande. Por debajo de 0.1 es prácticamente ruido.

In [ ]:
def d_de_cohen(x, y):
    """Diferencia de medias en unidades de desviación estándar combinada."""
    nx, ny = len(x), len(y)
    s = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2))
    return (x.mean() - y.mean()) / s if s > 0 else 0.0


enfermos = df[df[LABEL] == 1]
sanos    = df[df[LABEL] == 0]

filas = []
for col in NUMERICAS:
    filas.append({
        "variable": col,
        "media_sano": sanos[col].mean(),
        "media_alzheimer": enfermos[col].mean(),
        "mediana_sano": sanos[col].median(),
        "mediana_alzheimer": enfermos[col].median(),
        "min": df[col].min(),
        "max": df[col].max(),
        "diferencia_medias": enfermos[col].mean() - sanos[col].mean(),
        "efecto_d": d_de_cohen(enfermos[col], sanos[col]),
    })

resumen_num = pd.DataFrame(filas).round(3)
resumen_num = resumen_num.reindex(
    resumen_num["efecto_d"].abs().sort_values(ascending=False).index
).reset_index(drop=True)

guardar_tabla(resumen_num, "resumen_numericas_por_grupo")
resumen_num

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
colores = ["#DD8452" if v > 0 else "#4C72B0" for v in resumen_num["efecto_d"]]
ax.barh(resumen_num["variable"], resumen_num["efecto_d"], color=colores)
for umbral in [-0.5, -0.2, 0.2, 0.5]:
    ax.axvline(umbral, color="grey", ls=":", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.invert_yaxis()
ax.set(xlabel="d de Cohen (positivo = mayor en Alzheimer)", ylabel="",
       title="Tamaño de efecto de las variables numéricas")
plt.tight_layout()
guardar_figura("efecto_numericas")
plt.show()

---
## 4. Tabla resumen de variables binarias

Aquí no hace falta estandarizar: todas están en la misma escala de 0 a 100%.
El criterio de orden es la **diferencia de prevalencia en puntos porcentuales**.

In [ ]:
filas = []
for col in BINARIAS:
    p_sano = sanos[col].mean()
    p_alz  = enfermos[col].mean()
    filas.append({
        "variable": col,
        "n_sano": int(sanos[col].sum()),
        "n_alzheimer": int(enfermos[col].sum()),
        "prev_sano_pct": p_sano * 100,
        "prev_alzheimer_pct": p_alz * 100,
        "diferencia_puntos": (p_alz - p_sano) * 100,
        "razon_prevalencias": (p_alz / p_sano) if p_sano > 0 else np.nan,
    })

resumen_bin = pd.DataFrame(filas).round(3)
resumen_bin = resumen_bin.reindex(
    resumen_bin["diferencia_puntos"].abs().sort_values(ascending=False).index
).reset_index(drop=True)

guardar_tabla(resumen_bin, "resumen_binarias_por_grupo")
resumen_bin

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colores = ["#DD8452" if v > 0 else "#4C72B0" for v in resumen_bin["diferencia_puntos"]]
ax.barh(resumen_bin["variable"], resumen_bin["diferencia_puntos"], color=colores)
ax.axvline(0, color="black", lw=0.8)
ax.invert_yaxis()
ax.set(xlabel="Diferencia de prevalencia (puntos porcentuales, Alzheimer - sano)",
       ylabel="", title="Antecedentes y hábitos: diferencia entre grupos")
plt.tight_layout()
guardar_figura("efecto_binarias")
plt.show()

---
## 5. Distribuciones de las variables numéricas

Un histograma y un boxplot por variable, separados por diagnóstico. Se guardan todos,
pero a la presentación van solo 2 o 3.

In [ ]:
for col in NUMERICAS:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

    sns.histplot(data=df, x=col, hue="diagnostico", palette=PALETA,
                 kde=True, common_norm=False, stat="density",
                 element="step", fill=False, ax=ax[0])
    ax[0].set(title=f"Distribución de {col}", ylabel="Densidad")

    sns.boxplot(data=df, x="diagnostico", y=col, hue="diagnostico",
                palette=PALETA, legend=False, ax=ax[1])
    d = resumen_num.loc[resumen_num["variable"] == col, "efecto_d"].iloc[0]
    ax[1].set(title=f"{col} por diagnóstico (d = {d:.2f})", xlabel="")

    plt.tight_layout()
    guardar_figura(f"{col}_distribucion", "distribuciones")
    plt.close(fig)

print(f"{len(NUMERICAS)} figuras guardadas en {DIR_FIGURAS / 'distribuciones'}")

In [ ]:
# Panel compacto con las 4 numéricas de mayor efecto (candidata a la presentación)
top4 = resumen_num["variable"].head(4).tolist()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, top4):
    sns.boxplot(data=df, x="diagnostico", y=col, hue="diagnostico",
                palette=PALETA, legend=False, ax=ax)
    d = resumen_num.loc[resumen_num["variable"] == col, "efecto_d"].iloc[0]
    ax.set(title=f"{col}\nd = {d:.2f}", xlabel="")
plt.tight_layout()
guardar_figura("panel_numericas_top")
plt.show()

---
## 6. Prevalencias de las variables binarias

In [ ]:
prev = (resumen_bin
        .melt(id_vars="variable",
              value_vars=["prev_sano_pct", "prev_alzheimer_pct"],
              var_name="grupo", value_name="prevalencia")
        .replace({"prev_sano_pct": "Sano", "prev_alzheimer_pct": "Alzheimer"}))

fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=prev, y="variable", x="prevalencia", hue="grupo",
            palette=PALETA, ax=ax)
ax.set(xlabel="Prevalencia (%)", ylabel="",
       title="Prevalencia de antecedentes y hábitos por diagnóstico")
ax.legend(title="")
plt.tight_layout()
guardar_figura("prevalencias_binarias", "categoricas")
plt.show()

In [ ]:
# Panel con las 5 binarias de mayor diferencia
top5_bin = resumen_bin["variable"].head(5).tolist()

sub = prev[prev["variable"].isin(top5_bin)]
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=sub, x="variable", y="prevalencia", hue="grupo", palette=PALETA, ax=ax)
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.0f%%", padding=2, fontsize=9)
ax.set(xlabel="", ylabel="Prevalencia (%)",
       title="Antecedentes con mayor diferencia entre grupos")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="")
plt.tight_layout()
guardar_figura("panel_binarias_top", "categoricas")
plt.show()

---
## 7. Correlación entre variables numéricas

Dos usos distintos:

1. **Descriptivo** — qué variables se mueven juntas.
2. **Operativo** — detectar redundancia antes del PCA. Variables con |r| ≥ 0.8 miden
   esencialmente lo mismo, y si entran todas al PCA sobrerrepresentan ese bloque.

In [ ]:
corr_total = df[NUMERICAS].corr()
corr_sano  = sanos[NUMERICAS].corr()
corr_alz   = enfermos[NUMERICAS].corr()

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (matriz, titulo) in zip(axes, [(corr_total, "Cohorte completa"),
                                       (corr_sano, "Sanos"),
                                       (corr_alz, "Alzheimer")]):
    sns.heatmap(matriz, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, vmin=-1, vmax=1, square=True,
                cbar=False, annot_kws={"size": 7}, ax=ax)
    ax.set_title(titulo)
plt.tight_layout()
guardar_figura("matrices_correlacion", "correlacion")
plt.show()

In [ ]:
UMBRAL_REDUNDANCIA = 0.80

pares = []
for i, a in enumerate(NUMERICAS):
    for b in NUMERICAS[i + 1:]:
        r = corr_total.loc[a, b]
        if abs(r) >= UMBRAL_REDUNDANCIA:
            pares.append({"variable_1": a, "variable_2": b, "r": round(r, 3)})

redundantes = pd.DataFrame(pares)

if len(redundantes):
    efecto = resumen_num.set_index("variable")["efecto_d"].abs()
    redundantes["efecto_1"] = redundantes["variable_1"].map(efecto)
    redundantes["efecto_2"] = redundantes["variable_2"].map(efecto)
    redundantes["se_conserva"] = np.where(
        redundantes["efecto_1"] >= redundantes["efecto_2"],
        redundantes["variable_1"], redundantes["variable_2"])
    redundantes["se_descarta"] = np.where(
        redundantes["efecto_1"] >= redundantes["efecto_2"],
        redundantes["variable_2"], redundantes["variable_1"])
    guardar_tabla(redundantes, "pares_redundantes")
    display(redundantes)
    print("Candidatas a descartar por redundancia:",
          sorted(set(redundantes["se_descarta"])))
else:
    print(f"No hay pares con |r| >= {UMBRAL_REDUNDANCIA}")

**Cómo leerlo:** varias variables antropométricas describen lo mismo. Además, `bmi` se
calcula a partir de `height_cm` y `weight_kg`, y `waist_hip_ratio` a partir de `waist_cm` y
`hip_cm`: son variables **derivadas**, no mediciones independientes. Eso hay que decirlo en
la defensa, porque explica la correlación sin necesidad de invocar biología.

---
## 8. Selección de las 3–5 variables clínicas

Criterio explícito y defendible:

1. Descartar las redundantes detectadas en el paso anterior.
2. Ordenar el resto por tamaño de efecto.
3. Tomar las de mayor efecto, cuidando cubrir dominios clínicos distintos
   (edad, adiposidad, presión, antecedentes).

In [ ]:
descartadas = set(redundantes["se_descarta"]) if len(redundantes) else set()

candidatas_num = [v for v in resumen_num["variable"] if v not in descartadas]
print("Numéricas ordenadas por efecto (sin redundantes):")
for v in candidatas_num:
    d = resumen_num.loc[resumen_num["variable"] == v, "efecto_d"].iloc[0]
    print(f"   {v:<20} d = {d:+.3f}")

print("\nBinarias ordenadas por diferencia de prevalencia:")
for _, fila in resumen_bin.head(6).iterrows():
    print(f"   {fila['variable']:<26} {fila['diferencia_puntos']:+.1f} puntos")

In [ ]:
# Propuesta automática: las de mayor efecto que no fueron descartadas por redundancia.
# EDITAR si quieres forzar otra combinación (p. ej. cubrir un dominio clínico distinto).
VARIABLES_NUMERICAS_SEL = candidatas_num[:3]
VARIABLES_BINARIAS_SEL  = resumen_bin["variable"].head(2).tolist()

print("Numéricas seleccionadas:", VARIABLES_NUMERICAS_SEL)
print("Binarias seleccionadas: ", VARIABLES_BINARIAS_SEL)

seleccion = pd.concat([
    resumen_num[resumen_num["variable"].isin(VARIABLES_NUMERICAS_SEL)]
        .assign(tipo="numerica", criterio="tamaño de efecto (d de Cohen)")
        [["variable", "tipo", "criterio", "efecto_d"]]
        .rename(columns={"efecto_d": "magnitud"}),
    resumen_bin[resumen_bin["variable"].isin(VARIABLES_BINARIAS_SEL)]
        .assign(tipo="binaria", criterio="diferencia de prevalencia (puntos)")
        [["variable", "tipo", "criterio", "diferencia_puntos"]]
        .rename(columns={"diferencia_puntos": "magnitud"}),
], ignore_index=True)

guardar_tabla(seleccion, "variables_seleccionadas")
seleccion

### Redacción para la presentación

> Reemplazar los valores por los que entregue la tabla anterior.

Las variables con mayor diferencia entre grupos fueron **edad** (d = ..., el efecto más grande
de la cohorte), **presión sistólica** (d = ...) y, entre los antecedentes, **antecedente familiar
de Alzheimer** y **antecedente de hipertensión**, con diferencias de ... y ... puntos porcentuales.
Las variables antropométricas (talla, peso, BMI, perímetros) **no** mostraron diferencias
apreciables entre grupos, lo que también es un resultado: acota dónde buscar señal.

---
## 9. Verificación: ¿el sexo confunde las comparaciones?

El sexo condiciona talla, peso y perímetros corporales. Si los grupos estuvieran desbalanceados
por sexo, una diferencia antropométrica podría reflejar composición del grupo y no diagnóstico.

In [ ]:
tabla_sexo = pd.crosstab(df["diagnostico"], df["sex_female"], normalize="index").mul(100).round(1)
tabla_sexo.columns = ["Hombre (%)", "Mujer (%)"]
display(tabla_sexo)

dif_sexo = abs(tabla_sexo.loc["Alzheimer", "Mujer (%)"] - tabla_sexo.loc["Sano", "Mujer (%)"])
print(f"Diferencia en proporción de mujeres entre grupos: {dif_sexo:.1f} puntos")
print("Los grupos están razonablemente balanceados por sexo."
      if dif_sexo < 10 else
      "Atención: desbalance por sexo, interpretar con cuidado las variables antropométricas.")

# Comparación estratificada
estratificado = (df.groupby(["sex_female", "diagnostico"])[NUMERICAS]
                   .mean().round(2))
estratificado

---
## 10. Proyección PCA

El PCA se aplica **solo a variables numéricas estandarizadas y no redundantes**. Las binarias
quedan fuera: estandarizar un 0/1 no produce una distancia interpretable.

Es un método **no supervisado**: busca los ejes de máxima varianza, no los que separan sanos de
enfermos. Si los grupos no se separan, es un resultado válido, no un error.

In [ ]:
# Todas las numéricas no redundantes, hasta un máximo de 5.
# EDITAR si prefieres fijar la lista a mano.
VARIABLES_PCA = candidatas_num[:5]
print("Variables que entran al PCA:", VARIABLES_PCA)

X = df[VARIABLES_PCA]
X_esc = StandardScaler().fit_transform(X)

pca = PCA(random_state=RANDOM_STATE)
scores = pca.fit_transform(X_esc)

var_exp = pca.explained_variance_ratio_
print("Varianza explicada por componente (%):", np.round(var_exp * 100, 1))
print(f"Acumulada PC1 + PC2: {var_exp[:2].sum() * 100:.1f}%")

df_pca = pd.DataFrame(scores[:, :2], columns=["PC1", "PC2"])
df_pca["diagnostico"] = df["diagnostico"].values
df_pca["sex_female"]  = df["sex_female"].values

cargas = pd.DataFrame(pca.components_[:2].T, index=VARIABLES_PCA,
                      columns=["PC1", "PC2"]).round(3)
display(cargas)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

ax[0].bar(range(1, len(var_exp) + 1), var_exp * 100, color="#4C72B0")
ax[0].plot(range(1, len(var_exp) + 1), np.cumsum(var_exp) * 100, "o-", color="#C44E52")
ax[0].set(xlabel="Componente", ylabel="% de varianza",
          title="Varianza explicada (barras) y acumulada (línea)")

sns.scatterplot(data=df_pca, x="PC1", y="PC2", hue="diagnostico",
                palette=PALETA, alpha=0.6, s=25, ax=ax[1])
ax[1].set(xlabel=f"PC1 ({var_exp[0]*100:.1f}%)", ylabel=f"PC2 ({var_exp[1]*100:.1f}%)",
          title=f"Proyección PCA (PC1+PC2 = {var_exp[:2].sum()*100:.1f}%)")

sns.heatmap(cargas, annot=True, cmap="coolwarm", center=0, ax=ax[2])
ax[2].set_title("Cargas de las variables")

plt.tight_layout()
guardar_figura("pca_panel", "proyecciones")
plt.show()

In [ ]:
# Biplot: pacientes + flechas de las variables
fig, ax = plt.subplots(figsize=(7.5, 6.5))
sns.scatterplot(data=df_pca, x="PC1", y="PC2", hue="diagnostico",
                palette=PALETA, alpha=0.45, s=25, ax=ax)

escala = 3
for i, v in enumerate(VARIABLES_PCA):
    ax.arrow(0, 0, pca.components_[0, i] * escala, pca.components_[1, i] * escala,
             color="black", width=0.012, head_width=0.13, alpha=0.85)
    ax.text(pca.components_[0, i] * escala * 1.18,
            pca.components_[1, i] * escala * 1.18, v, fontsize=9, ha="center")

ax.axhline(0, color="grey", lw=0.5)
ax.axvline(0, color="grey", lw=0.5)
ax.set(xlabel=f"PC1 ({var_exp[0]*100:.1f}%)", ylabel=f"PC2 ({var_exp[1]*100:.1f}%)",
       title="Biplot PCA — variables clínicas seleccionadas")
plt.tight_layout()
guardar_figura("pca_biplot", "proyecciones")
plt.show()

In [ ]:
# ¿Los componentes separan los grupos? ¿O separan por sexo?
print("Efecto entre grupos sobre los componentes:")
for pc in ["PC1", "PC2"]:
    d = d_de_cohen(df_pca[df_pca["diagnostico"] == "Alzheimer"][pc],
                   df_pca[df_pca["diagnostico"] == "Sano"][pc])
    print(f"   {pc}: d = {d:+.3f}")

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.scatterplot(data=df_pca, x="PC1", y="PC2", hue="diagnostico",
                style=df_pca["sex_female"].map({0: "Hombre", 1: "Mujer"}),
                palette=PALETA, alpha=0.6, s=30, ax=ax)
ax.set(title="PCA: color = diagnóstico, forma = sexo")
plt.tight_layout()
guardar_figura("pca_por_sexo", "proyecciones")
plt.show()

**Cómo interpretarlo en la defensa.** Si los efectos sobre PC1 y PC2 son cercanos a cero,
la conclusión es que **ninguna combinación lineal de las variables clínicas separa la cohorte
por diagnóstico**. Eso no invalida el análisis: justifica incorporar las capas de DNA y proteína,
que es exactamente el argumento del proyecto.

Si los marcadores de hombres y mujeres quedan en zonas distintas del plano, el PCA está
capturando sexo antes que diagnóstico, y también hay que decirlo.

---
## 11. Figura resumen de la cohorte

Una sola figura para la diapositiva de resultados clínicos.

In [ ]:
fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.28)

ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(x=balance.index, y=balance["n_pacientes"], palette=PALETA, ax=ax1)
ax1.set(title="A. Balance de la cohorte", xlabel="", ylabel="Pacientes")

var_top = resumen_num["variable"].iloc[0]
ax2 = fig.add_subplot(gs[0, 1])
sns.boxplot(data=df, x="diagnostico", y=var_top, hue="diagnostico",
            palette=PALETA, legend=False, ax=ax2)
ax2.set(title=f"B. {var_top} (mayor efecto)", xlabel="")

var_nula = resumen_num["variable"].iloc[-1]
ax3 = fig.add_subplot(gs[0, 2])
sns.boxplot(data=df, x="diagnostico", y=var_nula, hue="diagnostico",
            palette=PALETA, legend=False, ax=ax3)
ax3.set(title=f"C. {var_nula} (sin diferencia)", xlabel="")

ax4 = fig.add_subplot(gs[1, :2])
sns.barplot(data=prev[prev["variable"].isin(top5_bin)],
            x="variable", y="prevalencia", hue="grupo", palette=PALETA, ax=ax4)
ax4.set(title="D. Antecedentes con mayor diferencia", xlabel="", ylabel="Prevalencia (%)")
ax4.tick_params(axis="x", rotation=15)
ax4.legend(title="")

ax5 = fig.add_subplot(gs[1, 2])
sns.scatterplot(data=df_pca, x="PC1", y="PC2", hue="diagnostico",
                palette=PALETA, alpha=0.5, s=18, legend=False, ax=ax5)
ax5.set(title="E. Proyección PCA")

fig.suptitle("Resumen del EDA clínico — cohorte sintética de Alzheimer",
             fontsize=14, y=0.98)
guardar_figura("resumen_eda_clinico")
plt.show()

---
## 12. Exportar el insumo para el notebook 04 (integración)

La integración se hace por `patient_id`. Este notebook entrega la capa clínica ya limpia y
con las variables seleccionadas identificadas, para que `04_data_integration.ipynb` solo tenga
que hacer el merge.

In [ ]:
columnas_integracion = ([COL_ID, LABEL]
                        + VARIABLES_NUMERICAS_SEL
                        + VARIABLES_BINARIAS_SEL
                        + ["sex_female"])
columnas_integracion = list(dict.fromkeys(columnas_integracion))

clinica_integracion = df[columnas_integracion].copy()

ruta = guardar_tabla(clinica_integracion, "clinica_para_integracion")
print("Guardado:", ruta)
print("Filas:", len(clinica_integracion), "| Columnas:", list(clinica_integracion.columns))
clinica_integracion.head()

---
## 13. Conclusiones del EDA clínico

> Completar con los valores reales una vez ejecutado el notebook. Máximo 5 puntos,
> en lenguaje simple, tal como pide la guía.

1. La cohorte tiene 1.000 pacientes, 657 controles y 343 casos (~34% de casos).
2. La **edad** es la variable clínica con mayor diferencia entre grupos (d = ...), con pacientes
   con Alzheimer en promedio ... años mayores.
3. Entre los antecedentes, ... y ... presentan las mayores diferencias de prevalencia
   (... y ... puntos porcentuales).
4. Las variables **antropométricas no diferencian** a los grupos (todas con |d| < ...), pese a
   ser 6 de las 10 numéricas disponibles.
5. El PCA sobre las variables clínicas seleccionadas **no separa** los grupos
   (PC1+PC2 explican ...% de la varianza, con efecto ≈ ... entre diagnósticos), lo que motiva
   incorporar la información genética y proteica.

### Limitaciones

- Los datos son **sintéticos**: las asociaciones observadas fueron introducidas por el generador
  y no constituyen evidencia clínica.
- El análisis es **descriptivo**. No se aplicaron pruebas de hipótesis ni corrección por
  comparaciones múltiples, y el tamaño de efecto no implica significancia estadística.
- Toda relación observada es **asociación, no causalidad**.
- Varias variables numéricas son derivadas de otras (BMI, índice cintura-cadera), por lo que
  no aportan información independiente.

### Preguntas de defensa que este notebook deja cubiertas

| Pregunta | Dónde está la respuesta |
|---|---|
| ¿Cuántos pacientes hay en cada grupo? | Sección 2 |
| ¿Por qué esas 3–5 variables y no otras? | Secciones 3, 4 y 8 (criterio por tamaño de efecto) |
| ¿Cómo trataron variables numéricas y binarias? | Secciones 3 y 4 (dos escalas distintas) |
| ¿Por qué el sexo no entró al PCA? | Secciones 9 y 10 |
| ¿Qué significa que dos variables estén correlacionadas? | Sección 7 (variables derivadas) |
| ¿El PCA separa los grupos? | Sección 10 |
| ¿Qué no pueden concluir? | Sección 13, limitaciones |
